# Embedding

Mit dem Embedding wandeln wir die Chunks in Vektoren um und speichern die Daten im Chunk.

- Model: deepset/gbert-large
- Input: products_chunked.jsonl
- Output: products_embedded.jsonl

In [13]:
import json
import random
import numpy as np

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('deepset/gbert-large')

No sentence-transformers model found with name deepset/gbert-large. Creating a new one with mean pooling.


## Daten vorbereiten

Die Texte und die Chunks müssen getrennt verarbeitet werden bzw. in zwei Arrays abgelegt werden, die am dann per Index wieder zusammengeführt werden. Da die Eingangsdaten bereits als Chunks strukturiert sind, genügt es die zu vektorisieren Tete zu entnehmen und die Embeddings danach wieder hinzuzufügen.

In [ ]:
chunks = []

with open('../data/processed/products_chunked.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

# Text only
texts = [chunk['document'] for chunk in chunks]

#print(texts)

['Der Kirsch LABO-288 ist ein Laborkühlschrank, der speziell für den Einsatz in medizinischen und wissenschaftlichen Einrichtungen entwickelt wurde. Als einer der führenden Hersteller von Kühl- und Gefriergeräten im Medizintechniksektor bietet Kirsch mit dem LABO-288 ein Gerät, das auf eine über 100-jährige Geschichte an Qualität und Zuverlässigkeit zurückblicken kann. Der Laborkühlschrank ist ideal für den Einsatz in Laboren, Kliniken und anderen medizinischen Einrichtungen geeignet, wo präzise Kühlung und Zuverlässigkeit von größter Bedeutung sind. Kirsch stellt sicher, dass alle Bauteile, ob Eigenproduktion oder eingekauft, Made in Germany sind, was für höchste Qualität und Langlebigkeit steht.', 'Der Kirsch LABO-288 Laborkühlschrank ist mit einer statisch belüfteten, geräuscharmen und energiesparenden Kältemaschine ausgestattet, die für 220-240 V Wechselstrom ausgelegt ist. Die Kältemaschine ist hermetisch gekapselt und servicefreundlich, was eine lange Lebensdauer und einfache War

## Embedding

Es werden alle Daten übergeben und in 16er-Schritten encodet. Progressbar ist for fun, Normalisieren ist Standard bei ChromaDB, glaube ich. Da wir die Daten nur für die Ähnlichkeitssuche benötigen ist die Länge und die darin enthaltene semantische Bedeutung nicht relevant.

In [ ]:
embeddings = model.encode(
    texts,
    batch_size = 16,
    show_progress_bar = True,
    normalize_embeddings = True,
    convert_to_numpy = True
)

Batches:   0%|          | 0/331 [00:00<?, ?it/s]

Batches: 100%|██████████| 331/331 [21:27<00:00,  3.89s/it]


## Zusammenfassen

WIr verbinden mit zip() die Chunks der originalen Datei mit den Embeddings vom Model.

In [19]:
for chunk, embedding in zip (chunks, embeddings):
    chunk['embedding'] = embedding.tolist()

with open('../data/processed/products_embedded.jsonl', 'w', encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

## Evaluieren der Embeddings

In [20]:
# Längenvergleich
assert len(embeddings) == len(chunks)

# Stichproben
#sample_idx = random.sample(range(len(embeddings)), int(len(embeddings) * 0.01))
sample_idx = [0]
for idx in sample_idx:
    text = chunks[idx]['document']
    embd = embeddings[idx]
    norm = np.linalg.norm(embd)
    test = model.encode([text], normalize_embeddings=True)[0]
    similarity = np.dot(embd, test)

    print(f"Index: {idx}")
    print(f"Text: {text[:60]}...")
    print(f"Shape: {embd}, Norm: {norm:.4f}")
    print(f"Similarity: {similarity:.8f}")

assert not np.any(np.isnan(embeddings))
assert not np.any(np.isinf(embeddings))

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")

Index: 0
Text: Der Kirsch LABO-288 ist ein Laborkühlschrank, der speziell f...
Shape: [ 0.02153323 -0.00290169  0.0015596  ... -0.01071382 -0.02147422
 -0.01398658], Norm: 1.0000
Similarity: 0.92013794
Shape: (5293, 1024)
Dtype: float32
